<style>
table {
  margin-left: 0 !important;
  margin-right: auto !important;
}
th, td {
  text-align: left !important;
}
</style>


## 02-1 · Part 5: Case E, Simulation, and Uncertainty

**One cooling decision can be optimized for one weather case, for an average across cases, or for the worst listed case.**

Parts 1–4 formulated four everyday cases. This part returns to the familiar classroom system from 01-1 and formulates the three weather rules identified in 01-2.

### 1 · First classify function and response structure

The advertising case in Part 2 is linear because its scalar objective and constraints are linear in its continuous decision variables:

> $\displaystyle \underset{x}{\operatorname{minimize}}\quad c^{\mathsf T}x
\qquad\text{subject to}\qquad Ax\le b,\quad Cx=d.$

If an objective or constraint is nonlinear in the optimization variables, the formulation is nonlinear. Squares, products, ratios, \(\max\), and nonlinear response mappings are common sources.

| Case | Function structure | Response evaluation |
|:---|:---|:---|
| B · Advertising | Linear | Direct algebraic |
| D · Study | Nonlinear because \(Q\) contains ratios | Direct algebraic |
| E · Classroom | Nonlinear because \(D\) and \(E\) contain squared terms | Simulation-based |

For the classroom, \(\operatorname{Sim}\) repeats the physical transition \(F\) for \(n=12\) steps and then applies \(G\) to calculate \(D\) and \(E\). The scalar objective \(f(y;\lambda_E)\) is the score supplied by \(H\):

> $\displaystyle f(y;\lambda_E)=J(u;\lambda_E)=D(u)+\lambda_EE(u).$

Simulation-based does not mean uncertain. A simulation with one fixed external-input path is deterministic.

### 2 · Formulate Case E under three weather rules

<div style="text-align: left; margin: 0.65rem 0 1.5rem 0;">
  <img src="https://raw.githubusercontent.com/sonamu-jun/system-design-and-optimization/main/02-1_problem_formulation/assets/case_e_classroom.png" alt="A classroom air conditioner with Early and Late cooling controls and three outdoor weather scenarios of 29, 31, and 33 degrees Celsius." width="570" style="display: block; max-width: 100%; height: auto; margin: 0;">
</div>

Case E from 01-2: \(x=[u_{\mathrm{early}},u_{\mathrm{late}}]^{\mathsf T}\) is evaluated under the same three weather scenarios. The formulation states how their outcomes are compared.

The decision vector is \(x=[u_{\mathrm{early}},u_{\mathrm{late}}]^{\mathsf T}\in[0,5]^2\). It expands to \(u_0,\ldots,u_{11}\) as in 01-1. For weather scenario \(\xi_s\), the response is

> $\displaystyle y_s=\operatorname{Sim}(x;\xi_s)
=\begin{bmatrix}T_1^{(s)},\ldots,T_{12}^{(s)},D_s(x),E(x)\end{bmatrix}^{\mathsf T}.$

The scenarios use constant outdoor temperatures 29, 31, and 33 °C. Their stated probabilities are \(p=(0.2,0.5,0.3)\).

For scenario \(s\), define \(J_s(x;\lambda_E)=D_s(x)+\lambda_EE(x)\). The three formulations use different rules to combine the scenario scores:

| Rule | Scalar objective |
|:---|:---|
| Deterministic | \(f_{\mathrm{det}}(x)=J_{31}(x;\lambda_E)\) |
| Stochastic | \(f_{\mathrm{sto}}(x)=\sum_{s=1}^{3}p_sJ_s(x;\lambda_E)\) |
| Robust over the listed set | \(f_{\mathrm{rob}}(x)=\max_{s=1,2,3}J_s(x;\lambda_E)\) |

All three formulations use the cooling bounds and energy requirement. The deterministic formulation checks \(20\le T_t\le30\,^\circ\mathrm C\) for the 31 °C path. The stochastic and robust formulations check that safety range in all three listed scenarios. The probability-weighted objective and the worst-case objective still provide different comparison rules.

For the weather cases included by a rule, write the requirements as

> $\displaystyle E(x)-E_{\max}\le0,\qquad
T_{\min}-T_t^{(s)}\le0,\qquad T_t^{(s)}-T_{\max}\le0,$
>
> $\displaystyle t=1,\ldots,12,\qquad
(T_{\min},T_{\max},E_{\max})=(20,30,60).$

The deterministic rule fixes one external-input path. The stochastic rule uses probabilities. The robust rule protects against the worst case in the stated finite uncertainty set; it makes no claim about temperatures outside that set.

### 3 · Connect cooling decisions, weather responses, and comparison rules

The left panel simulates the same cooling decision under each outdoor temperature. Dashed horizontal lines mark the 20–30 °C requirements; the darker teal band marks the 22–24 °C comfort target.

The middle panel adds discomfort \(D_s\) and the energy penalty \(\lambda_E E\) to form each scenario score \(J_s\). The right panel applies the three objective rules to those scores. Orange bars are eligible current candidates; gray bars are rejected under that rule's weather coverage.

Each gold star is the best feasible score on the 0.5-unit cooling grid for its own rule. It may come from a different cooling decision than the current one shown in the temperature panel. Comparisons are made within each rule.

<div style="text-align: left; margin: 0.65rem 0 1.5rem 0;">
  <img src="https://raw.githubusercontent.com/sonamu-jun/system-design-and-optimization/main/02-1_problem_formulation/assets/case_e_formulation_graph.png" alt="Three linked panels show weather-dependent classroom temperatures, discomfort and weighted-energy scores, and the deterministic, stochastic, and robust objective comparisons." width="1000" style="display: block; max-width: 100%; height: auto; margin: 0;">
</div>

In [ ]:
import numpy as np

# Horizon and initial state
TIME_STEPS = 12
INITIAL_TEMPERATURE = 27.0

# Fixed parameters
WEATHER_EXCHANGE = 0.12
OCCUPANT_HEAT = 0.012
COOLING_EFFECT = 0.45

# External-input scenarios
OUTDOOR_SCENARIOS = np.array([29.0, 31.0, 33.0])
SCENARIO_PROBABILITIES = np.array([0.2, 0.5, 0.3])
OCCUPANTS = np.full(TIME_STEPS, 20.0)

# Requirement limits
MIN_COOLING, MAX_COOLING = 0.0, 5.0
MIN_TEMPERATURE, MAX_TEMPERATURE = 20.0, 30.0
MAX_ENERGY = 60.0


def expand_decision(x):
    early_cooling, late_cooling = np.asarray(x, dtype=float)
    return np.r_[np.full(6, early_cooling), np.full(6, late_cooling)]


def simulate_classroom(x, outdoor_temperature):
    cooling = expand_decision(x)
    temperatures = [INITIAL_TEMPERATURE]
    for people, action in zip(OCCUPANTS, cooling):
        current = temperatures[-1]
        temperatures.append(
            current
            + WEATHER_EXCHANGE * (outdoor_temperature - current)
            + OCCUPANT_HEAT * people
            - COOLING_EFFECT * action
        )
    return cooling, np.asarray(temperatures)


def performance_outputs(cooling, temperatures):
    discomfort = np.sum(
        np.maximum(temperatures[1:] - 24.0, 0.0) ** 2
        + np.maximum(22.0 - temperatures[1:], 0.0) ** 2
    )
    energy = 0.5 * np.sum(cooling**2)
    return float(discomfort), float(energy)


def evaluate_weather_rules(x, energy_weight=1.0):
    scenario_records = []
    for outdoor in OUTDOOR_SCENARIOS:
        cooling, temperatures = simulate_classroom(x, outdoor)
        discomfort, energy = performance_outputs(cooling, temperatures)
        scenario_records.append({
            "outdoor": float(outdoor),
            "temperatures": temperatures,
            "discomfort": discomfort,
            "energy": energy,
            "score": discomfort + float(energy_weight) * energy,
        })

    scores = np.array([record["score"] for record in scenario_records])
    energy = scenario_records[0]["energy"]
    common_requirements = (
        np.all(np.asarray(x) >= MIN_COOLING)
        and np.all(np.asarray(x) <= MAX_COOLING)
        and energy <= MAX_ENERGY
    )
    scenario_feasible = np.array([
        record["temperatures"][1:].min() >= MIN_TEMPERATURE
        and record["temperatures"][1:].max() <= MAX_TEMPERATURE
        for record in scenario_records
    ])
    return {
        "x": tuple(map(float, x)),
        "scenarios": scenario_records,
        "objectives": {
            "deterministic": float(scores[1]),
            "stochastic": float(SCENARIO_PROBABILITIES @ scores),
            "robust": float(scores.max()),
        },
        "feasible": {
            "deterministic": bool(common_requirements and scenario_feasible[1]),
            "stochastic": bool(common_requirements and np.all(scenario_feasible)),
            "robust": bool(common_requirements and np.all(scenario_feasible)),
        },
    }

In [ ]:
import sys
import matplotlib


def _pyplot(*, interactive=False):
    """Use the course's widget-backend fallback outside the browser runtime."""
    if interactive and sys.platform != "emscripten":
        try:
            matplotlib.use("widget", force=True)
        except (RuntimeError, ValueError):
            from matplotlib.backends import backend_registry

            backend_registry._clear()
            matplotlib.use("widget", force=True)
    import matplotlib.pyplot as plt
    return plt


BLUE = "#2878b5"
TEAL = "#168578"
ORANGE = "#e78b24"
PURPLE = "#8856a7"
GRAY = "#a6a6a6"
GOLD = "#f6c945"


def case_canvas(title, controls, *, panels=2, interactive=False):
    """Create a shared layout; controls are (name, label, low, high, value, step, color)."""
    plt = _pyplot(interactive=interactive)
    figure, axes = plt.subplots(
        1, panels, figsize=(12.4 if panels == 2 else 14.0, 7.4 if interactive else 5.3)
    )
    figure.suptitle(title, y=0.985, fontsize=14, fontweight="bold")
    status = figure.text(0.5, 0.918, "", ha="center", va="center", fontsize=11)
    footer = figure.text(0.5, 0.045 if not interactive else 0.27, "",
                         ha="center", va="center", fontsize=9)
    sliders = {}
    if interactive:
        from matplotlib.widgets import Slider

        figure.subplots_adjust(left=0.075, right=0.97,
                               bottom=0.41 if panels == 3 else 0.37, top=0.83, wspace=0.38)
        positions = np.linspace(0.19, 0.065, max(len(controls), 2))
        for position, (name, label, low, high, value, step, color) in zip(positions, controls):
            slider_axis = figure.add_axes([0.28, position, 0.61, 0.026])
            sliders[name] = Slider(slider_axis, label, low, high, valinit=value,
                                   valstep=step, valfmt="%1.0f" if step >= 1 else "%1.2f",
                                   color=color, initcolor=color)
    figure._case_sliders = sliders
    figure._case_state = {}
    return plt, figure, axes, sliders, status, footer


def finish_case(plt, figure, axes, sliders, refresh, *, interactive=False):
    """Connect controls and keep widgets and evaluated records alive on the figure."""
    for axis in axes:
        axis.grid(alpha=0.25)
    for slider in sliders.values():
        slider.on_changed(refresh)
    figure._case_refresh = refresh
    refresh()
    if not interactive:
        figure.tight_layout(rect=(0.015, 0.09, 0.985, 0.96))
    plt.show()
    if not interactive:
        plt.close(figure)
    return figure


def show_weather_formulation(decision=(3.0, 2.0), energy_weight=1.0, *, interactive=False):
    controls = [
        ("early", "Decision: early cooling", MIN_COOLING, MAX_COOLING, decision[0], 0.25, BLUE),
        ("late", "Decision: late cooling", MIN_COOLING, MAX_COOLING, decision[1], 0.25, BLUE),
        ("weight", "Hyperparameter: energy weight", 0.0, 3.0, energy_weight, 0.1, PURPLE),
    ]
    plt, figure, axes, sliders, status, footer = case_canvas(
        "Case E · One cooling decision, three weather paths, three comparison rules",
        controls, panels=3, interactive=interactive,
    )
    rules = ("deterministic", "stochastic", "robust")
    rule_labels = ("31 °C only", "Probability\naverage", "Worst listed\nweather")
    cooling_grid = np.arange(MIN_COOLING, MAX_COOLING + 0.25, 0.5)
    # Store physical responses once. The energy weight changes only their scores.
    baseline = [evaluate_weather_rules([early, late], energy_weight=0.0)
                for early in cooling_grid for late in cooling_grid]

    def weighted_record(record, weight):
        scores = np.array([s["discomfort"] + weight * s["energy"] for s in record["scenarios"]])
        return {
            **record,
            "scenarios": [{**scenario, "score": float(score)}
                          for scenario, score in zip(record["scenarios"], scores)],
            "objectives": dict(zip(rules, [float(scores[1]),
                                           float(SCENARIO_PROBABILITIES @ scores),
                                           float(scores.max())])),
        }

    axes[0].axhspan(MIN_TEMPERATURE, MAX_TEMPERATURE, color=TEAL, alpha=0.07)
    axes[0].axhspan(22, 24, color=TEAL, alpha=0.2, label="Comfort target: 22–24 °C")
    for limit in (MIN_TEMPERATURE, MAX_TEMPERATURE):
        axes[0].axhline(limit, color="#555555", linestyle="--", linewidth=1)
    axes[0].axvline(6, color=BLUE, linestyle=":", alpha=0.45)
    for midpoint, label in ((3, "Early (0–5)"), (9, "Late (6–11)")):
        axes[0].text(midpoint, 0.985, label, transform=axes[0].get_xaxis_transform(),
                     ha="center", va="top", fontsize=8, color=BLUE)
    temperature_lines = []
    for outdoor, color, style in zip(OUTDOOR_SCENARIOS,
                                      ("#76a7cb", BLUE, "#174a71"), (":", "-", "--")):
        line, = axes[0].plot(np.arange(TIME_STEPS + 1), np.zeros(TIME_STEPS + 1),
                              color=color, linestyle=style, linewidth=2,
                              label=f"Outdoor: {outdoor:.0f} °C")
        temperature_lines.append(line)
    axes[0].set(xlabel="Time step", ylabel="Indoor temperature (°C)",
                xlim=(0, TIME_STEPS), ylim=(12, 36), xticks=[0, 3, 6, 9, 12],
                title="Weather changes the state path")
    axes[0].legend(fontsize=7.5, loc="lower left")
    positions = np.arange(3)
    discomfort_bars = axes[1].bar(positions, np.zeros(3), color=ORANGE, label="Discomfort D")
    energy_bars = axes[1].bar(positions, np.zeros(3), color=PURPLE, label=r"Energy penalty $\lambda_E E$")
    score_labels = [axes[1].text(i, 0, "", ha="center", va="bottom", fontsize=9) for i in positions]
    axes[1].set(xticks=positions,
                xticklabels=[f"{outdoor:.0f} °C\np = {probability:.1f}"
                             for outdoor, probability in zip(OUTDOOR_SCENARIOS, SCENARIO_PROBABILITIES)],
                xlabel="Outdoor-temperature scenario", ylabel="Scenario score J (score units)",
                title="Each score combines D and energy")
    axes[1].legend(fontsize=8, loc="upper left")
    objective_bars = axes[2].bar(positions, np.zeros(3), color=ORANGE)
    best_marker = axes[2].scatter([], [], marker="*", s=190, color=GOLD,
                                  edgecolor="black", zorder=5, label="Best feasible grid score")
    objective_labels = [axes[2].text(i, 0, "", ha="center", va="bottom", fontsize=8)
                        for i in positions]
    axes[2].set(xticks=positions, xticklabels=rule_labels,
                ylabel="Objective f (score units)", title="Each rule compares eligible candidates")
    axes[2].legend(fontsize=8, loc="upper left")

    def refresh(_=None):
        values = (sliders["early"].val, sliders["late"].val) if sliders else decision
        weight = sliders["weight"].val if sliders else energy_weight
        current = evaluate_weather_rules(values, energy_weight=weight)
        weighted_grid = [weighted_record(record, weight) for record in baseline]
        selected = {
            rule: min((r for r in weighted_grid if r["feasible"][rule]),
                      key=lambda r: r["objectives"][rule])
            for rule in rules
        }
        scores = []
        for line, d_bar, e_bar, label, scenario in zip(
            temperature_lines, discomfort_bars, energy_bars, score_labels, current["scenarios"]
        ):
            line.set_ydata(scenario["temperatures"])
            d_bar.set_height(scenario["discomfort"])
            e_bar.set_y(scenario["discomfort"])
            e_bar.set_height(weight * scenario["energy"])
            scores.append(scenario["score"])
            label.set_y(scenario["score"])
            label.set_text(f"{scenario['score']:.1f}")
        axes[1].set_ylim(0, max(10.0, max(scores) * 1.4))
        objectives = [current["objectives"][rule] for rule in rules]
        benchmarks = [selected[rule]["objectives"][rule] for rule in rules]
        label_offset = 0.05 * max(10.0, max(objectives + benchmarks))
        best_marker.set_offsets(np.column_stack([positions, benchmarks]))
        for bar, label, rule, objective in zip(objective_bars, objective_labels, rules, objectives):
            eligible = current["feasible"][rule]
            bar.set_height(objective)
            bar.set_color(ORANGE if eligible else GRAY)
            label.set_y(objective + label_offset)
            label.set_text(f"{objective:.1f}\n{'eligible' if eligible else 'rejected'}")
        axes[2].set_ylim(0, max(10.0, max(objectives + benchmarks) * 1.5))
        energy = current["scenarios"][0]["energy"]
        status.set_text(f"Current cooling = ({values[0]:.2f}, {values[1]:.2f})"
                        f"   |   Energy E = {energy:.2f} / 60"
                        rf"   |   Energy weight $\lambda_E$ = {weight:.1f}")
        selections = "   |   ".join(
            f"{label}: ({selected[rule]['x'][0]:.1f}, {selected[rule]['x'][1]:.1f})"
            for label, rule in zip(("31 °C", "Average", "Worst"), rules)
        )
        footer.set_text(f"Best grid cooling (early, late) — {selections}\n"
                        "Orange: current eligible candidate; gray: rejected. Gold stars use a 0.5-unit cooling grid.\n"
                        "Temperature limits: 20–30 °C; energy limit: 60. Each rule has its stated weather coverage.")
        figure._case_state.update(current=current, selected=selected, energy_weight=weight)
        figure.canvas.draw_idle()

    return finish_case(plt, figure, axes, sliders, refresh, interactive=interactive)

The two blue trackbars change early and late cooling. Early cooling applies to actions \(u_0,\ldots,u_5\); late cooling applies to \(u_6,\ldots,u_{11}\). The temperature paths, discomfort, energy, feasibility, and objective values then update together.

The purple trackbar changes the energy-weight hyperparameter \(\lambda_E\). With cooling fixed, the temperature paths, \(D_s\), \(E\), and feasibility stay unchanged. The purple energy penalties, objective bars, and selected grid candidates can change.

In [ ]:
formulation_explorer = show_weather_formulation(
    decision=(3.0, 2.0), energy_weight=1.0, interactive=True
)

Each gold star reports the best feasible score on the stated 0.5-unit grid under that formulation's objective and weather coverage. The rules can select different candidates because they ask different questions. None of the grid results guarantees a continuous optimum.

### 4 · Classify all five cases on independent axes

| Case | Constraints | Domain | Objectives | Functions | Evaluation | Uncertainty |
|:---|:---|:---|:---|:---|:---|:---|
| A · Clock | Unconstrained | Continuous | Single | Nonlinear | Algebraic | Deterministic |
| B · Advertising | Generally constrained | Continuous | Single | Linear | Algebraic | Deterministic |
| C · Snacks | Generally constrained | Mixed | Single | Linear | Algebraic | Deterministic |
| D · Study | Generally constrained | Continuous | Multiple | Nonlinear | Algebraic | Deterministic |
| E · Classroom, one weather case | Generally constrained | Continuous | Single | Nonlinear | Simulated | Deterministic |
| E · Classroom, probability average | Generally constrained | Continuous | Single | Nonlinear | Simulated | Stochastic |
| E · Classroom, worst listed case | Generally constrained | Continuous | Single | Nonlinear | Simulated | Robust |

These axes are independent. A problem is not merely “continuous” or “nonlinear”; a complete classification combines every applicable label.

The function labels also combine with integer domains. Case C is a mixed-integer linear program (MILP). A continuous linear case is a linear program (LP), a continuous nonlinear case is a nonlinear program (NLP), and a nonlinear case with integer variables is a mixed-integer nonlinear program (MINLP).

### 5 · Separate the formulation from the search algorithm

A formulation states what is chosen, how responses are produced, which candidates are feasible, and how feasible candidates are compared. An algorithm states how the candidate set is searched.

Grid search, gradient-based methods, and evolutionary methods are algorithms. Changing the algorithm while keeping \(x,\mathcal X,\operatorname{Sim},f,g,h\) fixed does not change the optimization problem. Changing any formulation part creates a different problem even if the same algorithm is used.

### Takeaway

Formulate first, then classify along independent axes:

> **identify \(x,\mathcal X,\operatorname{Sim},f,g,h\) → compare objective values only among feasible candidates → classify constraints, domain, objectives, functions, evaluation, and uncertainty → choose a suitable search method**

The five cases use the same formulation logic even though their real decisions differ. A change in the decision affects the represented system. A hyperparameter, probability model, uncertainty set, or search grid changes the analysis and must be stated explicitly.